# cv_r1 インタラクティブデモ（Colab 起動版）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slp-hu/Style-Bert-VITS2/blob/layer-b-cadence-seq/colab/cv_r1_demo_colab.ipynb)

`demo/app.py`（話者マップ + 混合合成の Gradio デモ）を Colab から起動し、**共有リンク**を発行する。
モデル重みを公開せず（Drive の自分のモデルのまま）デモを人に触ってもらえる — 学会デモ・共同研究者
向けの一時公開はこれで足りる。恒久公開（HF Spaces）は `demo/README_space.md` を参照。

- share リンクは **72 時間 or ランタイム切断まで**有効
- スモークモデルでも動く（`TARGET = "smoke"`）が品質は動作確認レベル


In [ ]:
# ===== §0-1 マウント・前提チェック【毎回・ランタイム再起動後も最初に実行】=====
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import os, subprocess

DRIVE_BASE = Path("/content/drive/MyDrive/Style-Bert-VITS2")   # setup と同じ値（clone の置き場所）
assert DRIVE_BASE.exists(), f"★Drive に fork clone が無い: {DRIVE_BASE}（先に cv_r1_train_setup_colab.ipynb を実行）"
os.chdir(DRIVE_BASE); print("cwd:", Path.cwd())   # 依存インストールは requirements.txt をここから読む（後段でローカルへ移る）

br = subprocess.run(["git","rev-parse","--abbrev-ref","HEAD"], capture_output=True, text=True).stdout.strip()
assert br == "layer-b-cadence-seq", f"★branch が違う: {br} → git checkout layer-b-cadence-seq"
print("branch:", br)

g = subprocess.run(["nvidia-smi","-L"], capture_output=True, text=True)
assert g.returncode == 0 and "GPU" in g.stdout, "★GPU ランタイムでない → [ランタイム]→[ランタイムのタイプを変更]→GPU"
print(g.stdout.strip())


In [ ]:
# ===== §0-2 GPU 判定と依存インストール【毎セッション実行・再起動後も再実行（冪等）】=====
# GPU の compute capability で経路を自動分岐する:
#   ・sm_90 以下（T4/L4/A100 等）: requirements の pin どおり torch 2.3.1(cu121)。再起動不要。
#   ・sm_100 以上（Blackwell 系: RTX PRO 6000 = sm_120 等）: torch 2.3.1 は sm_90 までで非対応
#     （GPU forward で落ちる）→ torch 2.11.0+cu128 へ入替（07a 方式）。★入替後はランタイム再起動が必須。
# 再起動後にこのセルを再実行すると、入替をスキップして検証だけ行う。
import subprocess, sys, re, os
from pathlib import Path

def _run(cmd, stream=False):
    print("$", " ".join(cmd))
    if stream:   # 進捗をそのまま流す（★torch の数GB DL は -q だと無言=フリーズと誤認するため）
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout: print(line, end="")
        p.wait(); assert p.returncode == 0, f"失敗: {cmd}"
    else:
        r = subprocess.run(cmd, capture_output=True, text=True)
        print((r.stdout or "")[-1200:] or "(quiet)")
        if r.returncode != 0:
            print(r.stderr[-4000:]); raise SystemExit(f"失敗: {cmd}")

# --- GPU capability（torch を import せず nvidia-smi で判定するのが肝）---
cap = subprocess.run(["nvidia-smi","--query-gpu=compute_cap","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip().splitlines()
assert cap and cap[0], "★GPU が見えない（GPU ランタイムか確認）"
CAP = float(cap[0]); IS_BLACKWELL = CAP >= 10.0
print(f"compute capability: {cap[0]} (sm_{int(CAP*10)}) → 経路: "
      + ("Blackwell（torch 2.11+cu128 入替）" if IS_BLACKWELL else "標準（torch 2.3.1 のまま）"))

# --- 現在の torch を subprocess で確認（in-kernel import しない = 再起動不要性を保つ）---
q = subprocess.run([sys.executable,"-c","import torch;print(torch.__version__)"], capture_output=True, text=True)
TORCH_NOW = q.stdout.strip()
print("現在の torch:", TORCH_NOW or "(未導入)")
ALREADY_SWAPPED = IS_BLACKWELL and TORCH_NOW.startswith("2.11.")

if ALREADY_SWAPPED:
    print("→ 再起動後の再実行と判断: torch 入替済みのため install をスキップ")
else:
    # (1) requirements から faster-whisper を除外して install（07a 方式）。
    #     faster-whisper==0.10.1 が av==10.* をソースビルドしようとし Py3.12 で失敗するため。
    #     文字起こし用途で cv_r1（bert_gen/style_gen/train/eval）には不要。
    req = Path("requirements.txt"); req_train = Path("requirements_no_whisper.txt")
    _drop = re.compile(r"^(faster-whisper|av)==")   # ビルドで詰まるパッケージが増えたらここに追加（例: stable_ts）
    kept = [l for l in req.read_text(encoding="utf-8").splitlines() if not _drop.match(l.strip())]
    req_train.write_text("\n".join(kept) + "\n", encoding="utf-8")
    _run([sys.executable,"-m","pip","install","-q","-r",str(req_train)])

    # (2) Blackwell のみ: torch 2.11.0+cu128 へ入替（実績のある手順をそのまま踏む）
    if IS_BLACKWELL:
        _run([sys.executable,"-m","pip","uninstall","-y",
              "torch","torchaudio","torchvision","torchcodec","torchao","torchtune","torchdata"])
        _run([sys.executable,"-m","pip","install","torch==2.11.0","torchaudio==2.11.0",
              "--index-url","https://download.pytorch.org/whl/cu128"], stream=True)   # ★-q 禁止
        _run([sys.executable,"-m","pip","install","-q","soundfile"])
        _run([sys.executable,"-m","pip","uninstall","-y",
              "torchcodec","torchvision","torchao","torchtune","torchdata"])

    # (3) HF スタック固定（transformers 未ピン → Colab 既定の transformers 5.x が torch>=2.4 を要求して
    #     torch を無効化し、bert_gen が "AutoModelForMaskedLM requires PyTorch" で落ちる問題の恒久対策）
    _run([sys.executable,"-m","pip","install","-q",
          "transformers==4.41.2","huggingface_hub==0.23.5","tokenizers<0.20",
          "pytorch-lightning==2.2.5","torchmetrics<1.5","pyannote.audio==3.1.1",
          "scipy==1.13.1","numpy==1.26.4"])

# --- 検証（別プロセス。transformers から torch が見えているかまで確認）---
v = subprocess.run([sys.executable,"-c",
    "import torch;from transformers.utils import is_torch_available;"
    "print('torch',torch.__version__,'| cuda',torch.cuda.is_available(),"
    "'| transformers_sees_torch',is_torch_available())"], capture_output=True, text=True)
print(v.stdout.strip() or v.stderr[-800:])
assert "transformers_sees_torch True" in v.stdout, "★transformers が torch を認識していない → このセルをやり直す"
if (not IS_BLACKWELL) or ALREADY_SWAPPED:
    assert "cuda True" in v.stdout, "★CUDA が使えない（GPU ランタイム / torch ビルドを確認）"

if IS_BLACKWELL and not ALREADY_SWAPPED:
    print()
    print("=" * 70)
    print("★torch を入れ替えた → ここで【ランタイム再起動】が必須★")
    print("  [ランタイム] → [セッションを再起動] のあと、§0-1 → §0-2 → §0-3 の順に再実行してから先へ進む。")
    print("  （再起動で cwd も torch 状態もリセットされる。最初のセルを飛ばさないこと）")
    print("=" * 70)


In [ ]:
# ===== §0-3 torchaudio / torch.load shim【Blackwell 経路のみ実体化・毎セッション実行】=====
# torch 2.11 系では torchaudio.set_audio_backend が削除され、torchaudio.load も torchcodec 経由で壊れる。
# pyannote.audio が import 時に set_audio_backend を呼ぶため、shim なしでは style_gen / 合成が落ちる。
# `!python` で走る bert_gen / style_gen / train は【別プロセス】なので、カーネル内 monkeypatch では効かない
# → sitecustomize.py + PYTHONPATH で全 Python プロセスに注入する（ここが肝）。
import os, subprocess, sys
from pathlib import Path

if not IS_BLACKWELL:
    print("標準経路（torch 2.3.1）: shim 不要 → スキップ")
else:
    compat = Path("/content/_compat"); compat.mkdir(exist_ok=True)
    shim = compat / "sitecustomize.py"
    SHIM_SRC = '# sitecustomize: torch 2.11 環境の互換 shim（cv_r1 公開ノート用）\n# 1) torch.load の weights_only 既定を False に戻す（旧 ckpt / torch.hub モデルの読込互換）\n# 2) torchaudio.set_audio_backend / get_audio_backend を復活（pyannote.audio の import 時呼び出し対策）\n# 3) torchaudio.load / info を soundfile 実装に置換（torchcodec 経由の破綻を回避）\ntry:\n    import torch\n    _orig_torch_load = torch.load\n    def _patched_torch_load(*args, **kwargs):\n        kwargs.setdefault("weights_only", False)\n        return _orig_torch_load(*args, **kwargs)\n    torch.load = _patched_torch_load\nexcept Exception:\n    pass\n\ntry:\n    import torchaudio\n\n    def _noop_set_backend(*args, **kwargs):\n        return None\n    def _get_backend(*args, **kwargs):\n        return "soundfile"\n    torchaudio.set_audio_backend = _noop_set_backend\n    torchaudio.get_audio_backend = _get_backend\n\n    def _sf_load(filepath, frame_offset=0, num_frames=-1, normalize=True,\n                 channels_first=True, format=None, buffer_size=4096, backend=None):\n        import soundfile as sf\n        import torch as _torch\n        frames = int(num_frames) if int(num_frames) > 0 else -1\n        data, sr = sf.read(str(filepath), start=int(frame_offset), frames=frames,\n                           dtype="float32", always_2d=True)\n        wav = _torch.from_numpy(data.T if channels_first else data)\n        return wav, sr\n    torchaudio.load = _sf_load\n\n    def _sf_info(filepath, format=None, buffer_size=4096, backend=None):\n        import soundfile as sf\n        info = sf.info(str(filepath))\n        class _AudioMetaData:\n            pass\n        meta = _AudioMetaData()\n        meta.sample_rate = info.samplerate\n        meta.num_frames = info.frames\n        meta.num_channels = info.channels\n        meta.bits_per_sample = 16\n        meta.encoding = "PCM_S"\n        return meta\n    torchaudio.info = _sf_info\nexcept Exception:\n    pass\n'
    shim.write_text(SHIM_SRC, encoding="utf-8")

    # 書き出し事故（末尾のエスケープ崩れ等 → SyntaxError）を必ず py_compile で検証する
    r = subprocess.run([sys.executable,"-m","py_compile",str(shim)], capture_output=True, text=True)
    assert r.returncode == 0, "★shim が SyntaxError: " + r.stderr[-600:]

    pp = os.environ.get("PYTHONPATH","")
    if str(compat) not in pp.split(":"):
        os.environ["PYTHONPATH"] = f"{compat}:{pp}" if pp else str(compat)
    print("PYTHONPATH:", os.environ["PYTHONPATH"])

    # 機能確認: 別プロセスで torchaudio.load が shim（_compat）実装に置換されているか
    chk = subprocess.run([sys.executable,"-c",
        "import torchaudio;torchaudio.set_audio_backend('soundfile');"
        "import inspect;print('shim OK:', inspect.getsourcefile(torchaudio.load))"],
        capture_output=True, text=True, env=os.environ.copy())
    print(chk.stdout.strip() or chk.stderr[-800:])
    assert "shim OK" in chk.stdout and "_compat" in chk.stdout, "★shim が別プロセスに効いていない"


In [ ]:
# ===== §0-4 model_assets 接続（データセット展開は不要）=====
import os
from pathlib import Path
ROOT = Path("/content/Style-Bert-VITS2"); os.chdir(ROOT)
DRIVE_BASE = Path("/content/drive/MyDrive/Style-Bert-VITS2")
!ln -sfn {DRIVE_BASE}/model_assets {ROOT}/model_assets
print("model_assets ->", os.readlink(ROOT/"model_assets"))


In [ ]:
# ===== §1 デモ起動（share リンク発行）=====
import os, sys, re, json, importlib
from pathlib import Path
ROOT = Path("/content/Style-Bert-VITS2"); os.chdir(ROOT); sys.path.insert(0, str(ROOT))

TARGET = "auto"   # "full" | "smoke" | "auto"（full 優先）

!pip install -q "gradio>=4.44"

# demo/app.py を確保（rsync 済みの repo に無ければ GitHub raw から取得）
app_py = ROOT/"demo"/"app.py"
if not app_py.exists():
    app_py.parent.mkdir(exist_ok=True)
    RAW = "https://raw.githubusercontent.com/slp-hu/Style-Bert-VITS2/layer-b-cadence-seq/demo/app.py"
    !wget -q -O {app_py} "{RAW}"
assert app_py.exists() and app_py.stat().st_size > 1000, "★demo/app.py を取得できない（fork へ commit 済みか確認）"

# 対象モデルの assets ディレクトリを決定して環境変数で app に渡す
_all  = list(Path("model_assets").glob("*/*_e*_s*.safetensors"))
_full  = [p for p in _all if p.parent.name != "cv_r1_smoke"]
_smoke = [p for p in _all if p.parent.name == "cv_r1_smoke"]
_pick = {"full": _full, "smoke": _smoke}.get(TARGET) or _full or _smoke
assert _pick, "★model_assets に safetensors が無い"
ASSETS = max(_pick, key=lambda p: int(re.search(r"_s(\d+)\.safetensors$", p.name).group(1))).parent
os.environ["MODEL_DIR"] = str(ASSETS)
print("MODEL_DIR =", ASSETS)

sys.path.insert(0, str(ROOT/"demo"))
import app as _app; importlib.reload(_app)
_app.demo.launch(share=True)   # ← 出力される public URL を配布する
